# Lektion 04 - Tool Use Design Pattern

In dieser Lektion lernen Sie das **Tool Use** Designmuster für KI-Agenten mit dem Microsoft Agent Framework (Python) kennen. Wir behandeln:

- Definition von Funktions-Tools mit dem `@tool` Dekorator und typisierten Parametern
- Bereitstellung von Tool-Schemas, damit das Modell weiß, wofür jedes Tool dient
- Steuerung der Tool-Ausführung mit `approval_mode`
- Rückgabe von **strukturierten Ausgaben** über Pydantic-Modelle und `response_format`

Das Szenario ist ein **Reisebuchungsagent**, der Reiseziele recherchieren, Verfügbarkeiten prüfen und Fluginformationen abrufen kann.


## Einrichtung


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Werkzeuge mit dem @tool-Dekorator definieren

Der `@tool`-Dekorator verwandelt eine einfache Python-Funktion in ein Werkzeug, das ein Agent aufrufen kann.  
Wichtige Punkte:

- Der **Docstring** wird zur Werkzeugbeschreibung, die das Modell sieht.  
- **Typannotationen** (einschließlich `Annotated` mit Beschreibungen) definieren das Werkzeugschema.  
- `approval_mode` steuert, ob der Benutzer jeden Aufruf genehmigen muss, bevor er ausgeführt wird.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Erstellen eines Agenten mit mehreren Werkzeugen

Übergeben Sie alle drei Werkzeuge an den Client, damit das Modell diejenigen aufrufen kann, die es benötigt, um die Frage des Benutzers zu beantworten.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturierte Ausgabe mit Tools

Indem `response_format` auf ein Pydantic-Modell gesetzt wird, wird der Agent gezwungen, ein gut typisiertes JSON-Objekt anstelle von Freitext zurückzugeben. Dies ist nützlich, wenn nachfolgender Code das Ergebnis programmatisch verarbeiten muss.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Tool-Zulassungsmuster

Der Parameter `approval_mode` bei `@tool` steuert, ob Tool-Aufrufe vor der Ausführung eine menschliche Genehmigung erfordern:

| Modus | Verhalten |
|---|---|
| `"never_require"` | Tool läuft automatisch – keine Benutzerbestätigung erforderlich. |
| `"always_require"` | Jeder Aufruf muss vom Benutzer genehmigt werden, bevor er ausgeführt wird. |

Verwenden Sie `"always_require"` für Tools, die Nebenwirkungen haben (z. B. Flugbuchung, Belastung einer Kreditkarte), damit ein Mensch eingebunden bleibt.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie Sie:

1. **Werkzeuge definieren** mit dem `@tool` Dekorator, ausgestattet mit typisierten Parametern und Docstrings, die als Werkzeug-Schema dienen.
2. **Mehrere Werkzeuge kombinieren**, sodass der Agent sie nacheinander aufrufen kann, um komplexe Anfragen zu beantworten.
3. **Strukturierte Ausgaben zurückgeben**, indem ein Pydantic-Modell als `response_format` übergeben wird.
4. **Werkzeugfreigabe steuern** mit `approval_mode`, um bei sensiblen Vorgängen einen Menschen in den Prozess einzubinden.

Diese Muster bilden die Grundlage für den Aufbau zuverlässiger, produktionsreifer Agenten, die sicher mit externen Systemen interagieren können.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner ursprünglichen Sprache gilt als maßgebliche Quelle. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir haften nicht für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
